In [ ]:
# Qwen3.5-0.8B Stage A2 evaluation (eval-only, cheap MC logprob + domain NLL)


In [ ]:
# Cell 0 — environment / P100 proof + mount check
import os, shutil, sys
from pathlib import Path

print('python', sys.version.split()[0], '(torch loads in Cell 1 with a P100-compatible build)')
!nvidia-smi --query-gpu=name,memory.total,memory.free,compute_cap --format=csv 2>/dev/null || nvidia-smi 2>/dev/null | head -20
print('working disk:', shutil.disk_usage('/kaggle/working'))
inp = sorted(str(p) for p in Path('/kaggle/input').glob('*')) if Path('/kaggle/input').exists() else []
print('/kaggle/input mounts:', inp if inp else 'NONE (real-PLE will abort until PLE dataset is attached)')
secret_value_0 = os.environ.get('HF_TOKEN')
secret_value_1 = os.environ.get('KAGGLE_API_TOKEN')
if not secret_value_0 or not secret_value_1:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        secret_value_0 = secret_value_0 or user_secrets.get_secret('HF_TOKEN')
        secret_value_1 = secret_value_1 or user_secrets.get_secret('KG_TOKEN')
    except Exception:
        pass
assert secret_value_0, 'set HF_TOKEN or attach the Kaggle HF_TOKEN secret'
assert secret_value_1, 'set KAGGLE_API_TOKEN or attach the Kaggle KG_TOKEN secret'
os.environ['HF_TOKEN'] = secret_value_0
os.environ['KAGGLE_API_TOKEN'] = secret_value_1


In [ ]:
# Cell 1 — deps (P100-compatible torch BEFORE first torch import)
import subprocess, sys
try:
    _cap=subprocess.run(['nvidia-smi','--query-gpu=compute_cap','--format=csv,noheader'],capture_output=True,text=True,timeout=60).stdout.strip().splitlines()[0].strip()
except Exception: _cap=''
print('compute_cap:', _cap or 'unknown')
%pip install -q "transformers==5.17.0" datasets safetensors huggingface_hub matplotlib accelerate
if _cap.startswith('6.'):
    subprocess.run([sys.executable,'-m','pip','install','-q','--index-url','https://download.pytorch.org/whl/cu118','torch==2.5.1+cu118'],check=True)
    print('pinned P100 torch (cu118, sm_60 kernels)')
    subprocess.run([sys.executable,'-m','pip','install','-q','--index-url','https://download.pytorch.org/whl/cu118','torchvision==0.20.1+cu118'],check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','--index-url','https://download.pytorch.org/whl/cu118','torchaudio==2.5.1+cu118'],check=True)
    print('pinned matching torchvision+torchaudio (qwen3_5 modeling pulls both via media utils)')
import torch
assert torch.cuda.is_available(), 'need a GPU accelerator'
torch.zeros(1).cuda()  # fail fast if the build lacks sm_60 kernels
print('torch', torch.__version__, '| cap', torch.cuda.get_device_capability(0))
import transformers, datasets, safetensors, huggingface_hub
print(transformers.__version__, datasets.__version__, safetensors.__version__, huggingface_hub.__version__)

In [ ]:
# Cell 2 — single config block (edit here only)
from dataclasses import dataclass

@dataclass(frozen=True)
class Cfg:
    SOURCE_ID: str = 'Qwen/Qwen3.8-Flash-Next-FP8'
    TARGET_ID: str = 'Qwen/Qwen3.5-0.8B'
    MEM_DIM: int = 2560
    HIDDEN: int = 1024
    N_LAYERS: int = 24
    NGRAM: int = 3
    HEADS_PER_NGRAM: int = 8   # 16 address slots per token
    ROW_DIM: int = 160         # per-slot row dim; 16*160=2560
    ROWS_PER_PART: int = 2_500_012
    VOCAB_BASE: int = 20_000_000
    SEED: int = 1234           # PLE hash seed — never change
    EOS: int = 248044  # config <|endoftext|>: PLE-training terminator (NOT chat <|im_end|> 248046)
    VOCAB: int = 248320
    SEQ: int = 512
    VAL_FAST_TOKENS: int = 65536    # 128 x 512
    VAL_FULL_TOKENS: int = 524288   # 1024 x 512 (35B parity)
    DATASET_ID: str = 'HuggingFaceFW/fineweb-edu'  # FIRST experiment: FineWeb-Edu ONLY
    DATASET_CONFIG: str = 'sample-10BT'
    SMOKE_TOKENS: int = 5120    # correctness pass only (10 x 512); sweep stays OFF
    LR: float = 3e-5
    WD: float = 0.01
    WARMUP_FRAC: float = 0.05
    CKPTS: tuple = (100000, 250000, 500000)  # threshold-crossing (not exact multiples)
    PLACEMENTS: tuple = ((2,), (8,), (2, 8))  # zero-based IDX, 35B convention
    BRANCHES: int = 1
    GAMMA_INIT: float = 1e-3   # 0.0 = identity test mode
    EVAL_SEED: int = 1234
    EVAL_BS: int = 8
    EVAL_TARGET_S: int = 7200  # 2h soft target: A2 is cheap MC logprob, not generative
    EVAL_HARD_S: int = 32400  # 9h hard guard with persist margin (well below 12h limit)
    EVAL_MC_N: int = 1000  # per MC task before auto-reduce (min 250)
    EVAL_MC_MIN: int = 250
    EVAL_DOMAIN_TOKENS: int = 32768  # 64 x 512 per domain
    ARMS_500K: tuple = ('disabled', 'random', 'permuted', 'real')  # controlled comparison
    ARM_1M: str = 'real-1m'  # exploratory only, never in the causal set

C = Cfg()
def describe(layers): return f"IDX {list(layers)} = HUMAN {[l+1 for l in layers]}"
print(C)
print('placements:', ' / '.join(describe(l) for l in C.PLACEMENTS))
for sites, R in [(1,1),(2,1),(1,4),(2,4)]:
    n = sites*(R*C.MEM_DIM*C.HIDDEN + C.MEM_DIM*C.HIDDEN) + sites*R + sites
    print(f'sites={sites} R={R}: ~{n/1e6:.2f}M trainable')
print('controlled arms @500K:', C.ARMS_500K, '| exploratory:', C.ARM_1M)
print('A2: HellaSwag/PIQA/ARC-E/LAMBADA logprob + 5-domain NLL; cheap suite only')


In [ ]:
# Cell 3 — EXACT source addressing (port of src/qwen36_ple/hashing.py — do not modify)
import math
import torch

MASK64=(1<<64)-1; GAMMA=0x9E3779B97F4A7C15; M1=0xBF58476D1CE4E5B9; M2=0x94D049BB133111EB; LPRIME=10007

def splitmix64(v):
    v=(v+GAMMA)&MASK64; v=((v^(v>>30))*M1)&MASK64; v=((v^(v>>27))*M2)&MASK64; return (v^(v>>31))&MASK64

def layer_multipliers(vocab, ngram, ple_idx=0, seed=1234):
    mmax=((1<<63)-1)//max(vocab,1); hb=max(1,mmax//2); base=seed+LPRIME*ple_idx
    return tuple(2*(splitmix64((base+GAMMA*(i+1))&MASK64)%hb)+1 for i in range(ngram))

def is_prime(v):
    if v<2: return False
    if v%2==0: return v==2
    return all(v%d for d in range(3, math.isqrt(v)+1, 2))

def nth_prime_after(s, k):
    p=s
    for _ in range(k):
        p+=1
        while not is_prime(p): p+=1
    return p

def head_layout(ngram=3, hpn=8, base=20_000_000, ple_idx=0):
    n=(ngram-1)*hpn
    sizes=tuple(nth_prime_after(base-1, ple_idx*n+h+1) for h in range(n))
    off=[]; t=0
    for s in sizes: off.append(t); t+=s
    return sizes, tuple(off)

def shift_right_ignore_eos(ids, shift, eos):
    if shift==0: return ids
    B,L=ids.shape; pos=torch.arange(L, device=ids.device)
    eos_pos=torch.where(ids==eos, pos, -1); prev_inc=torch.cummax(eos_pos,1).values
    prev=torch.cat([eos_pos.new_full((B,1),-1), prev_inc[:,:-1]],1)
    inseg=pos.unsqueeze(0)-(prev+1); src=pos-shift
    sh=ids.gather(1, src.clamp_min(0).unsqueeze(0).expand(B,-1))
    valid=(inseg>=shift)&(src.unsqueeze(0)>=0)
    return torch.where(valid, sh, ids.new_full((), eos))

def ngram_indices(ids, eos_token_id=248044, vocab_size=248320, ngram_size=3, heads_per_ngram=8,
                    vocab_size_base=20_000_000, ple_layer_index=0, seed=1234):
    ids=ids.long()
    mult=torch.tensor(layer_multipliers(vocab_size, ngram_size, ple_layer_index, seed), device=ids.device)
    sizes, offs=head_layout(ngram_size, heads_per_ngram, vocab_size_base, ple_layer_index)
    sizes=torch.tensor(sizes, device=ids.device); offs=torch.tensor(offs, device=ids.device)
    sh=[shift_right_ignore_eos(ids,s,eos_token_id) for s in range(ngram_size)]
    blocks=[]
    for ng in range(2, ngram_size+1):
        st=(ng-2)*heads_per_ngram; mixed=sh[0]*mult[0]
        for p in range(1,ng): mixed=torch.bitwise_xor(mixed, sh[p]*mult[p])
        blocks.append(torch.remainder(mixed.unsqueeze(-1), sizes[st:st+heads_per_ngram])+offs[st:st+heads_per_ngram])
    return torch.cat(blocks,-1)  # [B,L,16] GLOBAL PLE addresses, never raw token ids

_a=ngram_indices(torch.tensor([[1,2,3,4,5]])); _b=ngram_indices(torch.tensor([[1,2,3,4,5]]))
assert torch.equal(_a,_b) and _a.shape==(1,5,16)
SIZES, OFFS = head_layout()
print('hash ok; slots/head addrs e.g.', tuple(_a[0,2,:4].tolist()), '| head0 range', (OFFS[0], OFFS[0]+SIZES[0]))
def addresses(token_cpu):
    '''ONLY path from tokens to PLE rows: exact source hashes -> global head addresses.'''
    return ngram_indices(token_cpu, eos_token_id=C.EOS, vocab_size=C.VOCAB,
                         ngram_size=C.NGRAM, heads_per_ngram=C.HEADS_PER_NGRAM,
                         vocab_size_base=C.VOCAB_BASE, ple_layer_index=0, seed=C.SEED)

In [ ]:
# Cell 4 — shared-value reader (hidden=1024; reductions stay FP32 so fp16 backbone is safe)
import math
import torch
from torch import nn

def rms_norm(x, eps=1e-6):  # always FP32 reduction, cast back: fp16-safe
    return x.float().mul(torch.rsqrt(x.float().square().mean(-1, keepdim=True)+eps)).to(x.dtype)

class SharedValueReader(nn.Module):
    def __init__(self, mem_dim=2560, hidden=1024, branches=1, gamma=0.0):
        super().__init__(); self.mem_dim=mem_dim; self.hidden=hidden; self.branches=branches
        self.keys=nn.ModuleList(nn.Linear(mem_dim, hidden, bias=False) for _ in range(branches))
        self.value=nn.Linear(mem_dim, hidden, bias=False)
        self.beta=nn.Parameter(torch.zeros(branches))
        self.gamma=nn.Parameter(torch.tensor(float(gamma)))
        self.last_gate=None
    def stats(self):
        if self.last_gate is None: return None
        g=self.last_gate.float()
        return {'mean':g.mean().item(),'std':g.std(correction=0).item(),'near_zero':(g<0.01).float().mean().item()}
    def forward(self, h, m):
        assert h.shape[:-1]==m.shape[:-1], (h.shape, m.shape)
        h_dtype=h.dtype; h=h.float(); m=m.float()  # backbone may be fp16; reader computes FP32
        q=rms_norm(h); v=self.value(m); gs=[]
        for b,proj in enumerate(self.keys):
            k=rms_norm(proj(m))
            s=(q.float()*k.float()).sum(-1)/math.sqrt(self.hidden)
            gs.append(torch.sigmoid(s+self.beta[b].float()).to(v.dtype))
        g=torch.stack(gs,0)
        o=(g.unsqueeze(-1)*v.unsqueeze(0)).mean(0)
        self.last_gate=g.detach()
        return (h+self.gamma*o).to(h_dtype)  # residual back to backbone dtype; params stay FP32

_r=SharedValueReader(gamma=0.0); _h=torch.randn(1,4,1024); _m=torch.randn(1,4,2560)
assert torch.equal(_r(_h,_m),_h)
print('reader identity ok; R=1 params:', sum(p.numel() for p in _r.parameters()))

In [ ]:
# Cell 5 — injection hooks (IDX convention) + layer helper
import torch
from torch import nn

def decoder_layers(model):
    for path in ['model.layers','language_model.layers','transformer.h']:
        o=model
        try:
            for a in path.split('.'): o=getattr(o,a)
            if len(o)==24 or len(o)>0: return o
        except Exception: pass
    raise RuntimeError('decoder layers not found')

class ReaderInjection(nn.Module):
    '''layers = zero-based IDX list, exactly like the 35B run (e.g. (2,) = third block).'''
    def __init__(self, model, layers, mem_dim=2560, hidden=1024, branches=1, gamma=0.0):
        super().__init__()
        self.idx=tuple(layers)
        self.readers=nn.ModuleDict({str(l):SharedValueReader(mem_dim,hidden,branches,gamma) for l in layers})
        self.memory=None; self.handles=[]
        dec=decoder_layers(model)
        assert len(dec)==C.N_LAYERS, f'decoder count {len(dec)} != {C.N_LAYERS} — wrong hook target'
        for l in layers:
            self.handles.append(dec[l].register_forward_pre_hook(self._hook(str(l)), with_kwargs=True))
        print(f'inject at IDX {list(layers)} = HUMAN {[l+1 for l in layers]}')
    def _hook(self,name):
        def fn(mod,args,kw):
            if self.memory is None: return args,kw
            m=self.memory
            L=args[0].shape[1] if args else kw['hidden_states'].shape[1]
            if m.shape[1]!=L: m=m[:,:L]
            if args: return (self.readers[name](args[0],m),*args[1:]),kw
            kw['hidden_states']=self.readers[name](kw['hidden_states'],m); return args,kw
        return fn
    def set_memory(self,m): self.memory=m
    def close(self):
        [h.remove() for h in self.handles]; self.handles.clear()

print('injection ok')

In [ ]:
# Cell 6 — PLE stores: real (mount-only, exact scale or abort) + calibrated controls
import json, os
from pathlib import Path
import torch
from safetensors import safe_open

PLE_TMPL='model.language_model.layers.1.ple.ple_embedding.ngram_embedding.shard_{p}.weight'
PLE_SCALE='model.language_model.layers.1.ple.ple_embedding.ngram_embedding.weight_scale'

def rss_mb():
    '''Host RSS in MiB (Linux /proc; -1 if unavailable). Proves bounded RAM.'''
    try:
        with open('/proc/self/status') as _f:
            for _line in _f:
                if _line.startswith('VmRSS:'): return float(_line.split()[1])/1024
    except Exception: pass
    try:
        import resource; return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1024
    except Exception: return -1.0

def find_ple_manifests():
    hits=sorted(Path('/kaggle/input').glob('*/manifest.json'))+sorted(Path('/kaggle/input').glob('*/*/manifest.json'))
    out=[]
    for h in hits:
        try:
            m=json.loads(h.read_text())
            if isinstance(m, dict) and 'parts' in m: out.append(h)
        except Exception: pass
    return out

class MountPLE:
    '''Real frozen PLE (row-level mmap, 35B-validated). REQUIRES /kaggle/input mount. Never downloads. Never caches parts: each lookup fetches ONLY needed rows via safetensors get_slice runs, so host RAM stays bounded after all 128 parts are touched.'''
    def __init__(self, manifest=None):
        manifests=[Path(manifest)] if manifest else find_ple_manifests()
        if not manifests or not all(p.exists() for p in manifests):
            raise RuntimeError('Real-PLE ABORT: no /kaggle/input PLE dataset attached. Attach the pinned shards+manifest.json dataset(s) first; refusing to download 48.7 GiB into /kaggle/working.')
        self.part_paths={}; revs=set()
        for mp in manifests:
            m=json.loads(mp.read_text())
            if not revs: self.rpp=m.get('rows_per_part',2500012); self.rd=m.get('row_dim',160)
            revs.add(m.get('ple_revision','unknown'))
            for k,v in m['parts'].items():
                p=mp.parent/v
                if not p.exists(): continue  # each dataset holds only its own shards
                if int(k) in self.part_paths:
                    assert self.part_paths[int(k)].name==p.name, f'part {k} filename clash'
                    continue
                self.part_paths[int(k)]=p
        assert len(revs)==1, f'mixed PLE revisions: {revs}'
        self.ple_revision=revs.pop()
        assert len(self.part_paths)==128, f'need all 128 parts, have {len(self.part_paths)}'
        missing=[str(p) for p in self.part_paths.values() if not p.exists()]
        if missing: raise RuntimeError(f"Real-PLE ABORT: {len(missing)} shard files missing, e.g. {missing[0]}")
        self.scale=self._resolve_scale()  # exact scale or abort — no fallback constant
        self.calls=0; self.rows_read=0
        self.parts_touched=set()  # cumulative DISTINCT parts; no part tensors ever held
        print(f'mounted PLE rev={self.ple_revision} parts={len(self.part_paths)} scale={self.scale}')
    def _resolve_scale(self):
        for f in sorted(set(self.part_paths.values())):
            try:
                with safe_open(f, framework='pt', device='cpu') as fh:
                    if PLE_SCALE in fh.keys():
                        return float(fh.get_tensor(PLE_SCALE).float().mean())
            except Exception: pass
        raise RuntimeError('Real-PLE ABORT: weight_scale tensor not found in pinned shards/index. Refusing hardcoded fallback.')
    def stats(self):
        return {'calls': self.calls, 'rows_read': self.rows_read,
                'parts_touched': len(self.parts_touched), 'held_part_tensors': 0,
                'scale': self.scale, 'rss_MiB': round(rss_mb(), 1)}
    @staticmethod
    def tensor_name(part): return PLE_TMPL.format(p=part)
    def lookup(self, indices):  # indices = GLOBAL head addresses [..,16] from ngram_indices()
        '''Row-level mmap reads: group deduped addresses by part (sorted), fetch ONLY
        needed rows as contiguous get_slice runs, dequantize the gathered rows. Full
        part tensors (~381 MiB each) are never materialized or cached.'''
        shape=indices.shape; flat=indices.detach().cpu().long().reshape(-1)
        uniq, inv=torch.unique(flat, return_inverse=True)  # dedup repeated addresses
        parts=torch.div(uniq, self.rpp, rounding_mode='floor'); local=uniq%self.rpp
        table=torch.empty(uniq.numel(), self.rd, dtype=torch.float32)
        for p in torch.unique(parts).tolist():  # ascending part order (page-friendly)
            pos=torch.nonzero(parts==p).flatten()
            lrows=local.index_select(0, pos); srows, sidx=torch.sort(lrows)
            runs=[]; a=int(srows[0]); prev=a
            for r in srows[1:].tolist():
                if r==prev+1: prev=r
                else: runs.append((a, prev+1)); a=prev=r
            runs.append((a, prev+1))
            path=self.part_paths.get(p)
            if path is None: raise FileNotFoundError(f'PLE part {p} not in manifest')
            with safe_open(str(path), framework='pt', device='cpu') as fh:
                sl=fh.get_slice(self.tensor_name(p))
                got=torch.cat([sl[x:y].to(torch.float32) for x, y in runs])*self.scale
            table.index_copy_(0, pos.index_select(0, sidx), got)  # got aligns with srows
        self.calls+=1; self.rows_read+=uniq.numel(); self.parts_touched.update(torch.unique(parts).tolist())
        return table[inv].reshape(*shape, self.rd).flatten(-2)  # [..,16,160]->[..,2560]

def permute_addresses(addrs, seed=777):
    '''Deterministic per-head bijective address permutation (shared by builder + store).'''
    sizes, offs=head_layout()
    S=torch.tensor(sizes); O=torch.tensor(offs)
    A=[]; B=[]
    for h,(s,o) in enumerate(zip(sizes, offs)):
        a=int(splitmix64((seed+10007*(h+1))&MASK64)%(s-1))+1
        b=int(splitmix64(((seed^0x9E3779B97F4A7C15)+7919*(h+1))&MASK64)%s)
        assert a%s!=0, 'A must be coprime to prime head size'
        A.append(a); B.append(b)
    A=torch.tensor(A); B=torch.tensor(B)
    f=addrs.long().cpu()
    H=f.shape[-1]
    O=O[:H]; A=A[:H]; B=B[:H]; S=S[:H]
    return O+((f-O)*A+B)%S

class RandomPLE:
    '''Calibrated per-head deterministic control: same global address -> same 160-d row, every call.
    Means/stds are per-head scalars measured from real rows in the frozen working set.
    16 rows concatenate to 2560-d. No 0.06 fallback.'''
    def __init__(self, head_means, head_stds, seed=0, row_dim=160):
        assert len(head_means)==16 and len(head_stds)==16, 'need 16 per-head stats'
        self.means=[float(m) for m in head_means]
        self.stds=[float(s) for s in head_stds]
        assert all(s>0 for s in self.stds), 'stds must be positive (calibrated)'
        self.seed=seed; self.rd=row_dim
        _sizes,_offs=head_layout()
        self._S=list(_sizes); self._O=list(_offs)
    def _head_of(self, a):
        for h in range(16):
            if self._O[h]<=a<self._O[h]+self._S[h]: return h
        raise ValueError('address outside head ranges')
    def _rows_for(self, uniq, heads):
        rows=[]
        for a,h in zip(uniq.tolist(), heads.tolist()):
            g=torch.Generator(); g.manual_seed((self.seed*1000003+int(a))%2**63)
            rows.append(torch.randn(self.rd, generator=g)*self.stds[h]+self.means[h])
        return torch.stack(rows)
    def lookup(self, indices):
        shape=indices.shape; flat=indices.detach().cpu().long().reshape(-1)
        uniq, inv=torch.unique(flat, return_inverse=True)
        heads=torch.tensor([self._head_of(int(a)) for a in uniq.tolist()])
        table=self._rows_for(uniq, heads)
        return table[inv].reshape(*shape, self.rd).flatten(-2)

def calibrate_head_stats(compact, n_per_head=4096, seed=0):
    '''Measure per-head mean/std from representative real rows in the compact working set.
    Samples n_per_head rows per head from the frozen prefix (train + full-val).'''
    sizes, offs=head_layout()
    addrs=compact.addrs.to(torch.int64)
    g=torch.Generator().manual_seed(seed)
    means=[]; stds=[]
    for h in range(16):
        lo=offs[h]; hi=offs[h]+sizes[h]
        pos=torch.nonzero((addrs>=lo)&(addrs<hi)).flatten()
        assert len(pos)>=n_per_head, f'head {h} only {len(pos)} rows'
        pick=pos[torch.randint(0,len(pos),(n_per_head,),generator=g)]
        rows=compact.rows.index_select(0, pick).to(torch.float32)*compact.scale
        means.append(float(rows.mean()))
        stds.append(float(rows.std(correction=0)))
    print('calibrated head means:', [round(m,6) for m in means])
    print('calibrated head stds:', [round(s,6) for s in stds])
    return means, stds

class PermutedPLE:
    '''Bijective per-head permutation: off_h + ((a-off_h)*A_h + B_h) % size_h.
    Preserves head ranges (sizes are prime, A_h % size_h != 0 so gcd=1 i.e. coprime) and table distribution.
    Wraps a compact working-set cache, never /kaggle/input random reads during training.'''
    def __init__(self, base, seed=777):
        self.b=base; self.seed=seed
        sizes, offs=head_layout()
        self.S=torch.tensor(sizes); self.O=torch.tensor(offs)
        A=[]; B=[]
        for h,(s,o) in enumerate(zip(sizes, offs)):
            a=int(splitmix64((seed+10007*(h+1))&MASK64)%(s-1))+1
            b=int(splitmix64(((seed^0x9E3779B97F4A7C15)+7919*(h+1))&MASK64)%s)
            assert a%s!=0, 'A must be coprime to prime head size'
            A.append(a); B.append(b)
        self.A=torch.tensor(A); self.B=torch.tensor(B)
    def lookup(self, indices):
        H=indices.shape[-1]; f=indices.long().cpu()
        O=self.O[:H]; A=self.A[:H]; B=self.B[:H]; S=self.S[:H]
        return self.b.lookup((O+((f-O)*A+B)%S).to(indices.device) if indices.is_cuda else (O+((f-O)*A+B)%S))

class CompactPLE:
    '''Compact working set for one frozen token prefix: sorted int32 addresses + fp8 rows (mmap).
    Bit-exact vs MountPLE: same bytes, same scale, same dequant formula. No 48.7 GiB traffic.'''
    def __init__(self, directory):
        import json as _json
        d=Path(directory)
        meta=_json.loads((d/'compact.json').read_text())
        n=meta['address_count']
        self.addrs=torch.from_file(str(d/'addrs.u32'), shared=True, size=n, dtype=torch.int32)
        raw=torch.from_file(str(d/'rows.u8'), shared=True, size=n*meta['row_dim'], dtype=torch.uint8)
        self.rows=raw.view(torch.float8_e4m3fn).view(n, meta['row_dim'])
        self.scale=float(meta['scale']); self.meta=meta
        self.ple_revision=meta.get('ple_revision')
        print('compact PLE: %d rows, %.2f GiB mapped, scale=%g' % (n, (d/'rows.u8').stat().st_size/1024**3, self.scale))
    def lookup(self, indices):
        shape=indices.shape; flat=indices.detach().cpu().long().reshape(-1)
        assert bool((flat>=0).all()) and int(flat.max())<2**31
        f32=flat.to(torch.int32)
        pos=torch.searchsorted(self.addrs, f32)
        posc=pos.clamp(max=len(self.addrs)-1)
        assert bool((self.addrs[posc]==f32).all()), 'compact miss: address outside frozen prefix'
        out=self.rows.index_select(0, posc).to(torch.float32)*self.scale
        return out.reshape(*shape, self.rows.shape[-1]).flatten(-2)



print('stores ok; manifests:', [str(p) for p in find_ple_manifests()])


In [ ]:
# Cell 7 — tokenizer verification: SEMANTIC id-space check (SPEC #15)
# Pinned-rev forensics: both model.vocab = 248044 entries with 0 id diffs; source-only
# ids are 7 audio added-tokens (248070-248076) above the target max: no collision.
# Config eos = 248044 (<|endoftext|>) on BOTH. AutoTokenizer.eos_token_id may report
# chat <|im_end|> 248046 instead: a chat-template default, NOT the PLE-training
# terminator. Hashing keeps training constants (vocab 248320 / eos 248044 / seed 1234).
import json
from transformers import AutoTokenizer
from huggingface_hub import HfApi, hf_hub_download

tok=secret_value_0; assert tok, 'Attach HF_TOKEN in Settings -> Secrets'
api=HfApi(token=tok)
trev=api.model_info(C.TARGET_ID).sha; srev=api.model_info(C.SOURCE_ID).sha
print('target rev', trev[:12], '| source rev', srev[:12])
tt=AutoTokenizer.from_pretrained(C.TARGET_ID, token=tok, revision=trev)
st=AutoTokenizer.from_pretrained(C.SOURCE_ID, token=tok, revision=srev)
def raw_vocab(path):
    tj=json.load(open(path, encoding='utf-8'))
    m=dict(tj['model']['vocab'])
    for a in tj.get('added_tokens', []): m[a['content']]=a['id']
    return m
tm=raw_vocab(hf_hub_download(C.TARGET_ID,'tokenizer.json',revision=trev,token=tok))
sm=raw_vocab(hf_hub_download(C.SOURCE_ID,'tokenizer.json',revision=srev,token=tok))
diff={k for k in tm if k in sm and tm[k]!=sm[k]}
extra_t={k for k in tm if k not in sm}
smax=max(tm.values())
src_only={k: sm[k] for k in sm if k not in tm}
print('mapping diffs:', len(diff), '| target-only:', len(extra_t), '| source-only:', len(src_only))
assert not diff and not extra_t, 'target id space must match source ids exactly (native addressing)'
assert all(v>smax for v in src_only.values()), 'source-only ids must sit above target range'
assert sm.get('<|endoftext|>')==C.EOS and tm.get('<|endoftext|>')==C.EOS, 'training eos must be <|endoftext|> both sides'
probe='The quick brown fox 0123456789 function print(){} <|im_start|>x<|im_end|>'
assert tt.encode(probe, add_special_tokens=False)==st.encode(probe, add_special_tokens=False), 'probe encodings differ — STOP'
print('NATIVE token-ID addressing VALID: identical ids; training terminator eos =', C.EOS)


In [ ]:
# Cell 8 — immutable validation artifacts (built ONCE, never inside training) + FineWeb-Edu-only stream
import hashlib, json
from array import array
from pathlib import Path
from datasets import load_dataset

WORK=Path('/kaggle/working/ple-08b'); WORK.mkdir(parents=True, exist_ok=True)
VALDIR=WORK/'val-frozen-v1'; VALDIR.mkdir(exist_ok=True)

def _write_val(name, tokens_u32, meta_extra):
    tp=VALDIR/f'tokens-{name}.uint32le'; mp=VALDIR/f'validation-{name}.json'
    raw=tokens_u32.tobytes(); digest=hashlib.sha256(raw).hexdigest()
    meta={'name':name,'token_count':len(tokens_u32),'seq':512,'count':len(tokens_u32)//512,
            'tokens_sha256':digest, **meta_extra}
    if mp.exists():
        old=json.loads(mp.read_text())
        if old!=meta or tp.read_bytes()!=raw:
            raise RuntimeError(f'Immutable validation {name} differs — refusing to overwrite')
        print(f"reuse frozen val-{name} sha={digest[:16]} n={len(tokens_u32)}"); return meta
    tp.write_bytes(raw); mp.write_text(json.dumps(meta,indent=2)); print(f'wrote frozen val-{name} sha={digest[:16]}')
    return meta

def build_validation_artifacts(tok):
    '''Prefix-consistent: stream 524288 FineWeb-Edu tokens once; fast = first 65536 slice.'''
    from huggingface_hub import HfApi
    import os
    api=HfApi(token=secret_value_0)
    drev=api.dataset_info(C.DATASET_ID).sha; trev2=api.model_info(C.TARGET_ID).sha
    need_full=VALDIR/'validation-full.json'; need_fast=VALDIR/'validation-fast.json'
    if need_full.exists() and need_fast.exists():
        return json.loads(need_fast.read_text()), json.loads(need_full.read_text())
    ds=load_dataset(C.DATASET_ID, C.DATASET_CONFIG, split='train', streaming=True, revision=drev)
    arr=array('I'); docs=0
    for row in ds:
        docs+=1; t=row.get('text') or ''
        if t.strip(): arr.extend(tok.encode(t, add_special_tokens=False)); arr.append(C.EOS)  # training terminator
        if len(arr)>=C.VAL_FULL_TOKENS: del arr[C.VAL_FULL_TOKENS:]; break
    assert len(arr)==C.VAL_FULL_TOKENS, len(arr)
    base={'dataset':C.DATASET_ID,'config':C.DATASET_CONFIG,'dataset_rev':drev,'target_rev':trev2,'docs':docs,'skip_docs':docs}
    fast_arr=array('I', arr[:C.VAL_FAST_TOKENS])
    mf=_write_val('fast', fast_arr, base); mF=_write_val('full', arr, base)
    return mf, mF

def load_validation(name):
    m=json.loads((VALDIR/f'validation-{name}.json').read_text())
    raw=(VALDIR/f'tokens-{name}.uint32le').read_bytes()
    assert hashlib.sha256(raw).hexdigest()==m['tokens_sha256'], 'val checksum mismatch'
    a=array('I'); a.frombytes(raw)
    import torch
    return torch.tensor(a,dtype=torch.long).view(-1,512), m



print('build with build_validation_artifacts(tok); read with load_validation("fast"/"full")')

In [ ]:
# Cell 9 — frozen Qwen3.5-0.8B via full-config VLM-compat CausalLM (vision frozen, text path): float16 FIRST
import os, torch
from transformers import AutoConfig, AutoModelForCausalLM

tok=secret_value_0; assert tok, 'Attach HF_TOKEN in Settings -> Secrets'
from huggingface_hub import HfApi
trev=HfApi(token=tok).model_info(C.TARGET_ID).sha
cfg=AutoConfig.from_pretrained(C.TARGET_ID, token=tok, revision=trev, trust_remote_code=True)
assert getattr(cfg,'model_type',None)=='qwen3_5', getattr(cfg,'model_type',None)
tconf=cfg.text_config  # dims ONLY — never pass as config= (strips auto_map, breaks class resolution)
print('hidden',tconf.hidden_size,'layers',tconf.num_hidden_layers,'vocab',tconf.vocab_size,'arch',type(cfg).__name__)
assert tconf.hidden_size==1024 and tconf.num_hidden_layers==24
from transformers import AutoTokenizer
tokenizer=AutoTokenizer.from_pretrained(C.TARGET_ID, token=tok, revision=trev, trust_remote_code=True)

try:
    from transformers.models.qwen3_5 import Qwen3_5ForCausalLM as _Q
    print('direct qwen3_5 import ok:', _Q.__name__)
except Exception:
    import traceback; traceback.print_exc()
    raise RuntimeError('qwen3_5 modeling import failed — true cause above')
ACTIVE_DTYPE=None; model=None
for dt in [torch.float16, torch.float32]:
    try:
        m=AutoModelForCausalLM.from_pretrained(C.TARGET_ID, revision=trev, token=tok,
            trust_remote_code=True, device_map={'':0}, torch_dtype=dt, low_cpu_mem_usage=True)
        m.eval(); m.requires_grad_(False)
        assert sum(1 for p in m.parameters() if p.requires_grad)==0
        ids=tokenizer('The quick brown fox jumps over the lazy dog. '*8, return_tensors='pt').input_ids[:,:64].cuda()
        with torch.inference_mode():
            lg=m(input_ids=ids,use_cache=False).logits
        assert torch.isfinite(lg.float()).all(), 'non-finite logits'
        model=m; ACTIVE_DTYPE=dt; print(f'frozen load OK in {dt} footprint={m.get_memory_footprint()/1024**3:.2f} GiB')
        print('model class:', type(m).__name__, '| decoder blocks:', len(decoder_layers(m)))
        del ids, lg; break
    except Exception as e:
        print(f'{dt} rejected: {str(e)[:200]}')
        try: del m
        except Exception: pass
assert model is not None and ACTIVE_DTYPE is not None
print('backbone = ACTIVE_DTYPE:', ACTIVE_DTYPE)
print('reader params/compute = FP32 (AdamW FP32; norms/scores FP32)')
print('PLE dequant/output = FP32')
print('reader residual output is cast back to backbone dtype')

In [ ]:
# Cell 11b — validation prep: build immutable artifacts ONCE, then load both (idempotent)
mf, mfull = build_validation_artifacts(tokenizer)
val_fast, _ = load_validation("fast")
val_full, _ = load_validation("full")

print("validation ready")
print("fast:", mf["tokens_sha256"])
print("full:", mfull["tokens_sha256"])


In [ ]:
# Stage A2 harness: MC adapters (HellaSwag/PIQA/ARC-E/LAMBADA) + frozen domain-NLL builders.
# Logprob scoring only; no free generation in this notebook.
PRED_A2 = []
import json, re, subprocess, sys, time, zlib
import torch, torch.nn.functional as F
EVAL_SEED = 1234
EVAL_BS = 8
DOMAINS = ('general', 'code', 'math', 'scientific', 'multilingual')
def _ds_rev(ds_id):
    from huggingface_hub import HfApi
    try:
        return HfApi(token=secret_value_0).dataset_info(ds_id).sha
    except Exception:
        return None
def _load_split(ds_id, split, config=None):
    from datasets import load_dataset
    rev = _ds_rev(ds_id)
    kw = {'revision': rev} if rev else {}
    if config is None:
        return load_dataset(ds_id, split=split, **kw), rev, None
    return load_dataset(ds_id, config, split=split, **kw), rev, config
def _subset(rows, n, seed=EVAL_SEED):
    import random
    rows = list(rows)
    if len(rows) <= n:
        return rows
    rng = random.Random(seed)
    idx = list(range(len(rows)))
    rng.shuffle(idx)
    return [rows[i] for i in sorted(idx[:n])]
def _load_first(cands, split):
    errs = []
    for _cid, _cfg in cands:
        try:
            _ds, _rev, _ = _load_split(_cid, split, _cfg)
            print('dataset resolved: ' + _cid + ('/' + _cfg if _cfg else ''), flush=True)
            return _ds, _rev, _cid, _cfg
        except Exception as e:
            errs.append(_cid + ': ' + str(e)[:120])
    raise RuntimeError('no candidate resolved: ' + ' | '.join(errs))
def _hellaswag_rows(n):
    ds, rev, ds_id, cfg = _load_first([('Rowan/hellaswag', None), ('hellaswag', None)], 'validation')
    rows = []
    for r in ds:
        ctx = r.get('ctx') or ((r.get('ctx_a') or '') + ' ' + (r.get('ctx_b') or '')).strip()
        ends = list(r.get('endings') or [])
        try:
            lab = int(str(r.get('label')).strip())
        except Exception:
            continue
        if not ctx or len(ends) != 4:
            continue
        rows.append({'ctx': ctx, 'endings': ends, 'label': lab})
    rows = _subset(rows, n)
    return rows, {'dataset': ds_id, 'config': cfg, 'rev': str(rev), 'n': len(rows), 'scoring': 'logprob-sum+length-norm'}
def _piqa_rows(n):
    # Config-first: ybisk/piqa offers a plain_text subset (legacy script path is
    # rejected by datasets>=4 when no config is given). Direct-jsonl fallback last.
    try:
        ds, rev, ds_id, cfg = _load_first([('ybisk/piqa', 'plain_text'), ('piqa', 'plain_text'), ('ybisk/piqa', None), ('piqa', None)], 'validation')
        ds_id_out, rev_out = ds_id, str(rev)
    except Exception:
        import urllib.request
        from datasets import load_dataset as _ld
        base = 'https://yonatanbisk.com/piqa/data/'
        ds = _ld('json', data_files={'validation': base + 'dev.jsonl'}, split='validation')
        with urllib.request.urlopen(base + 'dev-labels.lst', timeout=120) as _r:
            _labs = _r.read().decode('utf-8').split()
        ds_id_out, rev_out, cfg = 'yonatanbisk.com/piqa/data', 'direct-jsonl', None
        print('dataset resolved: yonatanbisk.com/piqa/data (direct jsonl)', flush=True)
        rows = []
        for r, _lab in zip(ds, _labs):
            g = (r.get('goal') or '').strip()
            s = [(r.get('sol1') or '').strip(), (r.get('sol2') or '').strip()]
            try:
                lab = int(str(_lab).strip())
            except Exception:
                continue
            if not g or not s[0] or not s[1] or lab not in (0, 1):
                continue
            rows.append({'goal': g, 'sols': s, 'label': lab})
        rows = _subset(rows, n)
        return rows, {'dataset': ds_id_out, 'config': cfg, 'rev': rev_out, 'n': len(rows), 'scoring': 'logprob-sum+length-norm'}
    rows = []
    for r in ds:
        g = (r.get('goal') or r.get('question') or r.get('query') or '').strip()
        s1 = (r.get('sol1') or '').strip()
        s2 = (r.get('sol2') or '').strip()
        if (not s1 or not s2) and isinstance(r.get('choices'), list) and len(r.get('choices')) >= 2:
            s1, s2 = str(r['choices'][0]).strip(), str(r['choices'][1]).strip()
        try:
            lab = int(str(r.get('label', r.get('answer', ''))).strip())
        except Exception:
            continue
        if not g or not s1 or not s2 or lab not in (0, 1):
            continue
        rows.append({'goal': g, 'sols': [s1, s2], 'label': lab})
    rows = _subset(rows, n)
    return rows, {'dataset': ds_id, 'config': cfg, 'rev': str(rev), 'n': len(rows), 'scoring': 'logprob-sum+length-norm'}
def _arc_rows(n):
    ds, rev, ds_id, cfg = _load_first([('allenai/ai2_arc', 'ARC-Easy'), ('ai2_arc', 'ARC-Easy')], 'validation')
    rows = []
    for r in ds:
        q = (r.get('question') or '').strip()
        ch = r.get('choices')
        try:
            if isinstance(ch, dict) and 'text' in ch and 'label' in ch:
                texts = [str(t).strip() for t in ch['text']]
                labs = [str(v).strip() for v in ch['label']]
            elif isinstance(ch, list):
                texts = [str(c.get('text', '')).strip() for c in ch]
                labs = [str(c.get('label', '')) for c in ch]
            else:
                continue
            key = str(r.get('answerKey')).strip()
            li = labs.index(key)
        except Exception:
            continue
        if not q or len(texts) < 2:
            continue
        rows.append({'question': q, 'options': texts, 'label': li, 'answerKey': key})
    rows = _subset(rows, n)
    return rows, {'dataset': ds_id, 'config': cfg, 'rev': str(rev), 'n': len(rows), 'scoring': 'logprob-sum+length-norm'}
def _lambada_rows(n):
    ds, rev, ds_id, cfg = _load_first([('EleutherAI/lambada_openai', None), ('lambada', None)], 'test')
    rows = []
    for r in ds:
        t = (r.get('text') or '').strip()
        parts = t.split()
        if len(parts) < 10:
            continue
        rows.append({'context': ' '.join(parts[:-1]), 'target': parts[-1], 'text': t})
    rows = _subset(rows, n)
    return rows, {'dataset': ds_id, 'config': cfg, 'rev': str(rev), 'n': len(rows), 'scoring': 'teacher-forced last-word NLL+acc'}
def _domain_cands(domain):
    if domain == 'general':
        return [('wikitext', 'wikitext-103-raw-v1', 'test', 'text'), ('wikitext', 'wikitext-2-raw-v1', 'test', 'text')]
    if domain == 'code':
        return [('edward-io/starcoderdata-repo', None, 'train', 'code'), ('codeparrot/github-code-clean', None, 'train', 'code'), ('bigcode/the-stack-smol', None, 'train', 'code')]
    if domain == 'math':
        return [('openai/gsm8k', 'main', 'test', 'qa'), ('gsm8k', 'main', 'test', 'qa')]
    if domain == 'scientific':
        return [('allenai/sciq', None, 'test', 'sci'), ('sciq', None, 'test', 'sci')]
    if domain == 'multilingual':
        return [('wikipedia', '20220301.de', 'train', 'text'), ('HuggingFaceFW/fineweb-2', 'deu_Latn', 'train', 'text')]
    raise ValueError('unknown domain ' + domain)
def _extract_domain_text(domain, row):
    if domain in ('general', 'multilingual'):
        return (row.get('text') or '')
    if domain == 'code':
        return (row.get('content') or row.get('code') or row.get('text') or '')
    if domain == 'math':
        return ((row.get('question') or '') + '\n' + (row.get('answer') or ''))
    if domain == 'scientific':
        return ((row.get('support') or '') + '\n' + (row.get('question') or '') + '\n' + (row.get('correct_answer') or row.get('answer') or ''))
    return ''
def _write_domain(name, tokens_u32, meta_extra):
    from array import array as _array
    import hashlib as _hl
    ddir = WORK / 'domain-frozen-v1'
    ddir.mkdir(parents=True, exist_ok=True)
    tp = ddir / f'tokens-{name}.uint32le'
    mp = ddir / f'domain-{name}.json'
    raw = tokens_u32.tobytes()
    digest = _hl.sha256(raw).hexdigest()
    meta = {'name': name, 'token_count': len(tokens_u32), 'seq': 512, 'count': len(tokens_u32) // 512, 'tokens_sha256': digest, **meta_extra}
    if mp.exists():
        old = json.loads(mp.read_text())
        if old != meta or tp.read_bytes() != raw:
            raise RuntimeError(f'Immutable domain {name} differs — refusing to overwrite')
        print(f"reuse frozen domain-{name} sha={digest[:16]} n={len(tokens_u32)}")
        return meta
    tp.write_bytes(raw)
    mp.write_text(json.dumps(meta, indent=2))
    print(f'wrote frozen domain-{name} sha={digest[:16]}')
    return meta
def build_domain_blocks(tok, domain, n_tokens):
    from array import array as _array
    from datasets import load_dataset as _ld
    from huggingface_hub import HfApi as _Api
    assert domain in DOMAINS, domain
    ddir = WORK / 'domain-frozen-v1'
    ddir.mkdir(parents=True, exist_ok=True)
    mp = ddir / f'domain-{domain}.json'
    if mp.exists():
        return load_domain(domain)
    errs = []
    for ds_id, cfg, split, kind in _domain_cands(domain):
        try:
            api = _Api(token=secret_value_0)
            drev = api.dataset_info(ds_id).sha
            kw = {'revision': drev} if drev else {}
            if cfg is None:
                ds = _ld(ds_id, split=split, streaming=True, **kw)
            else:
                ds = _ld(ds_id, cfg, split=split, streaming=True, **kw)
            arr = _array('I')
            docs = 0
            for row in ds:
                docs += 1
                t = _extract_domain_text(domain, row)
                if t and t.strip():
                    arr.extend(tok.encode(t, add_special_tokens=False))
                    arr.append(C.EOS)
                if len(arr) >= n_tokens:
                    del arr[n_tokens:]
                    break
            assert len(arr) == n_tokens, (domain, len(arr))
            meta = _write_domain(domain, arr, {'dataset': ds_id, 'config': cfg, 'split': split, 'kind': kind, 'dataset_rev': drev, 'docs': docs})
            return load_domain(domain)
        except Exception as e:
            errs.append(f'{ds_id}: {str(e)[:140]}')
    raise RuntimeError('no domain candidate resolved for ' + domain + ': ' + ' | '.join(errs))
def load_domain(domain):
    import hashlib as _hl
    from array import array as _array
    ddir = WORK / 'domain-frozen-v1'
    m = json.loads((ddir / f'domain-{domain}.json').read_text())
    raw = (ddir / f'tokens-{domain}.uint32le').read_bytes()
    assert _hl.sha256(raw).hexdigest() == m['tokens_sha256'], 'domain checksum mismatch ' + domain
    a = _array('I')
    a.frombytes(raw)
    import torch as _torch
    return _torch.tensor(a, dtype=_torch.long).view(-1, 512), m
print('A2 harness ready (4 MC adapters + 5 frozen domain builders, logprob only)')


In [ ]:
# Stage A2 run: 4x500K controlled + exploratory REAL-1M. Eval only, no training gates.
# Cheap MC logprob (HellaSwag/PIQA/ARC-E/LAMBADA) + 5-domain held-out NLL.
# Identical prompts/tokenizer/precision/seeds for all arms. Cheap suite only.
import hashlib, json, re, subprocess, sys, time, zlib
import torch, torch.nn.functional as F
from pathlib import Path

T0 = time.perf_counter()
EOUT = Path('/kaggle/working/eval-stageA2'); EOUT.mkdir(parents=True, exist_ok=True)
# persists hellaswag.jsonl piqa.jsonl arc-easy.jsonl lambada.jsonl domain-*.jsonl per-example/per-block
EFI_MOUNT = Path('/kaggle/input/qwen-ple-reader-checkpoints')
DLDIR = Path('/kaggle/working/ckpt-dl'); DLDIR.mkdir(parents=True, exist_ok=True)
need_pairs = [('reader-control-random-r1-500224.pt', 'reader-control-random-r1-500224.run.json', True),
              ('reader-control-permuted-r1-500224.pt', 'reader-control-permuted-r1-500224.run.json', True),
              ('real-500k-r1.reader.safetensors', 'real-500k-r1.run.json', False),
              ('real-1m-r1.reader.safetensors', 'real-1m-r1.run.json', False)]
def _mount_pair_ok(pt, runf, need_sha):
    try:
        run = json.loads((EFI_MOUNT / runf).read_text())
        ok = isinstance(run, dict) and 'token_count' in run and (EFI_MOUNT / pt).exists()
        if need_sha:
            ok = ok and 'sha256' in run
        if not ok:
            print('mount pair invalid: ' + pt, flush=True)
        return ok
    except Exception as e:
        print('mount pair error: ' + pt + ' ' + str(e)[:120], flush=True)
        return False
if EFI_MOUNT.exists() and all(_mount_pair_ok(p, r, s) for p, r, s in need_pairs):
    CKDIR = EFI_MOUNT
    print('checkpoints: mounted dataset (schema-validated)', flush=True)
else:
    _env = dict(os.environ); _env['KAGGLE_API_TOKEN'] = secret_value_1
    for _f in ['protocol.json',
               'reader-control-random-r1-500224.pt', 'reader-control-random-r1-500224.run.json',
               'reader-control-random-r1-500224.metrics.json',
               'reader-control-permuted-r1-500224.pt', 'reader-control-permuted-r1-500224.run.json',
               'reader-control-permuted-r1-500224.metrics.json',
               'real-500k-r1.reader.safetensors', 'real-500k-r1.run.json', 'real-500k-r1.metrics.json',
               'real-1m-r1.reader.safetensors', 'real-1m-r1.run.json', 'real-1m-r1.metrics.json']:
        _r = subprocess.run([sys.executable, '-m', 'kaggle', 'datasets', 'download', '-d',
                             'ninnix/qwen-ple-reader-checkpoints', '-f', _f, '-p', str(DLDIR), '--force'],
                            capture_output=True, text=True, env=_env, timeout=1200)
        assert _r.returncode == 0 and (DLDIR / _f).exists(), 'checkpoint download failed: ' + _f
    CKDIR = DLDIR
    print('checkpoints: api download (mount missing)', flush=True)
timings = {}
def _mark(name):
    timings[name] = round(time.perf_counter() - T0, 1)
    print('[t=%ds] %s' % (timings[name], name), flush=True)

_proto = json.loads((CKDIR / 'protocol.json').read_text())
def _file_sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()
def _load_state(pt_name, run_name, expect_tokens):
    run = json.loads((CKDIR / run_name).read_text())
    h = _file_sha(CKDIR / pt_name)
    assert h == run['sha256'], 'bytes mismatch: ' + pt_name
    assert run.get('token_count') == expect_tokens, 'budget mismatch: ' + pt_name
    d = torch.load(str(CKDIR / pt_name), map_location='cpu')
    return d['reader'], h
def _load_sf(pt_name, run_name, expect_tokens):
    from safetensors.torch import load_file
    run = json.loads((CKDIR / run_name).read_text())
    assert run.get('token_count') == expect_tokens, 'budget mismatch: ' + run_name
    assert (CKDIR / pt_name).exists(), 'missing: ' + pt_name
    return load_file(str(CKDIR / pt_name), device='cpu'), 'e2e-checked-below'
_cksha = {}
_arms = []
def _mk(name, state, store):
    inj = ReaderInjection(model, [2, 8], C.MEM_DIM, C.HIDDEN, 1, C.GAMMA_INIT).to('cuda')
    if state is not None:
        inj.load_state_dict({k: v.to('cuda') for k, v in state.items()})
    _arms.append({'name': name, 'inj': inj, 'store': store})
    print('arm ready: ' + name, flush=True)

# Resolve MC rows (max N) + frozen domain blocks before building the union compact.
# Per-task fault tolerance: one unresolvable dataset must not kill the run.
MCN = C.EVAL_MC_N
_mc = {}
_mc_meta = {}
_mc_err = {}
def _try_mc(name, fn):
    try:
        rows, meta = fn(MCN)
        _mc[name] = rows
        _mc_meta[name] = meta
        print('MC %s: %d rows (%s)' % (name, len(rows), meta.get('dataset')), flush=True)
    except Exception as e:
        _mc[name] = []
        _mc_meta[name] = {'error': str(e)[:300]}
        _mc_err[name] = str(e)[:300]
        print('MC %s FAILED (skipped): %s' % (name, str(e)[:200]), flush=True)
_try_mc('hellaswag', _hellaswag_rows)
_try_mc('piqa', _piqa_rows)
_try_mc('arc-easy', _arc_rows)
_try_mc('lambada', _lambada_rows)
hs_rows, pi_rows, arc_rows, lam_rows = _mc['hellaswag'], _mc['piqa'], _mc['arc-easy'], _mc['lambada']
hs_meta, pi_meta, arc_meta, lam_meta = _mc_meta['hellaswag'], _mc_meta['piqa'], _mc_meta['arc-easy'], _mc_meta['lambada']
print('MC rows: hellaswag %d piqa %d arc-easy %d lambada %d' % (len(hs_rows), len(pi_rows), len(arc_rows), len(lam_rows)), flush=True)
dom_blocks = {}
dom_metas = {}
dom_err = {}
for _d in DOMAINS:
    try:
        _b, _m = build_domain_blocks(tokenizer, _d, C.EVAL_DOMAIN_TOKENS)
        dom_blocks[_d] = _b
        dom_metas[_d] = _m
        print('domain %s blocks %d sha %s' % (_d, _b.shape[0], _m['tokens_sha256'][:16]), flush=True)
    except Exception as e:
        dom_err[_d] = str(e)[:300]
        print('domain %s FAILED (skipped): %s' % (_d, str(e)[:200]), flush=True)
if all(len(v) == 0 for v in _mc.values()) and not dom_blocks:
    raise RuntimeError('A2: no MC task and no domain resolved; aborting')
_mark('datasets resolved')

def _tok_ids(s):
    return tokenizer(s, return_tensors='pt', add_special_tokens=False)['input_ids'][0]

# Collect scoring inputs as token sequences for union-compact coverage.
_seqs = []
for r in hs_rows:
    p = _tok_ids(r['ctx'])
    for e in r['endings']:
        t = _tok_ids(' ' + e.strip() if not e.startswith(' ') else e)
        _seqs.append(torch.cat([p, t]))
for r in pi_rows:
    p = _tok_ids(r['goal'])
    for s in r['sols']:
        t = _tok_ids(' ' + s.strip() if not s.startswith(' ') else s)
        _seqs.append(torch.cat([p, t]))
for r in arc_rows:
    p = _tok_ids(r['question'])
    for o in r['options']:
        t = _tok_ids(' ' + o.strip() if not o.startswith(' ') else o)
        _seqs.append(torch.cat([p, t]))
for r in lam_rows:
    p = _tok_ids(r['context'])
    t = _tok_ids(' ' + r['target'].strip() if not r['target'].startswith(' ') else r['target'])
    _seqs.append(torch.cat([p, t]))
for _d in dom_blocks:
    for _bi in range(dom_blocks[_d].shape[0]):
        _seqs.append(dom_blocks[_d][_bi])
print('union sequences: %d' % len(_seqs), flush=True)
_real_parts = []
_perm_parts = []
with torch.inference_mode():
    for _s in _seqs:
        _a = ngram_indices(_s.unsqueeze(0).cpu())
        _real_parts.append(_a.reshape(-1))
        _perm_parts.append(permute_addresses(_a, seed=777).reshape(-1))
_uniq_real = torch.unique(torch.cat(_real_parts))
_uniq_perm = torch.unique(torch.cat(_perm_parts))
print('union addresses: real %d perm %d' % (len(_uniq_real), len(_uniq_perm)), flush=True)

def _build_a2_compact(pdir, uniq, perm_seed):
    import hashlib as _hl
    from array import array as _array
    from safetensors import safe_open as _so
    uniq = uniq.long().cpu()
    _uh = _hl.sha256(_array('I', uniq.tolist()).tobytes()).hexdigest()
    need = True
    if (pdir / 'compact.json').exists():
        try:
            pm = json.loads((pdir / 'compact.json').read_text())
            need = not (pm.get('address_count') == len(uniq) and pm.get('uniq_sha256') == _uh
                        and pm.get('permutation_seed', None) == perm_seed
                        and pm.get('ple_revision') == _proto['ple_revision'])
            if not need:
                print('reuse compact: ' + str(pdir), flush=True)
        except Exception:
            need = True
    if need:
        _ms = MountPLE()
        assert _ms.ple_revision == _proto['ple_revision'], 'PLE revision drift'
        _uq = uniq.sort().values
        pdir.mkdir(parents=True, exist_ok=True)
        _af = (pdir / 'addrs.u32').open('wb'); _rf = (pdir / 'rows.u8').open('wb')
        _ha = _hl.sha256(); _hr = _hl.sha256(); _n = 0; _prev = -1
        _pa = torch.div(_uq, C.ROWS_PER_PART, rounding_mode='floor'); _lo = _uq % C.ROWS_PER_PART
        _bf = {}
        for _p in torch.unique(_pa).tolist():
            _bf.setdefault(str(_ms.part_paths[_p]), []).append(_p)
        for _fp in sorted(_bf):
            with _so(_fp, framework='pt', device='cpu') as _fh:
                for _p in sorted(_bf[_fp]):
                    _full = _fh.get_slice(MountPLE.tensor_name(_p))[:]
                    _pos = torch.nonzero(_pa == _p).flatten()
                    _au = _uq.index_select(0, _pos)
                    assert int(_au[0]) > _prev; _prev = int(_au[-1])
                    _ab = _array('I', _au.tolist()).tobytes()
                    _rb = bytes(_full.index_select(0, _lo.index_select(0, _pos)).view(torch.uint8).flatten().tolist())
                    _af.write(_ab); _rf.write(_rb); _ha.update(_ab); _hr.update(_rb); _n += _pos.numel()
        _af.close(); _rf.close()
        (pdir / 'compact.json').write_text(json.dumps({'format': 'qwen-ple-compact-a2', 'version': 1,
            'address_count': _n, 'uniq_sha256': _uh, 'row_dim': C.ROW_DIM, 'scale': float(_ms.scale),
            'ple_revision': _ms.ple_revision, 'permutation_seed': perm_seed,
            'addrs_sha256': _ha.hexdigest(), 'rows_sha256': _hr.hexdigest()}, indent=2))
        _cc = CompactPLE(pdir)
        _g = torch.Generator().manual_seed(0)
        _samp = _uq[torch.randint(0, len(_uq), (2048,), generator=_g)].reshape(128, 16)
        _d = (_cc.lookup(_samp) - _ms.lookup(_samp)).abs().max().item()
        print('a2-compact equivalence max|diff|: %g' % _d, flush=True)
        assert _d == 0.0
    return CompactPLE(pdir)

_cval = _build_a2_compact(WORK / 'compact-a2', _uniq_real, None)
_cperm = _build_a2_compact(WORK / 'compact-a2-perm777', _uniq_perm, 777)
del _real_parts, _perm_parts, _uniq_real, _uniq_perm, _seqs
_mark('union compacts ready')

_mk('disabled', None, None)
_st, _h = _load_state('reader-control-random-r1-500224.pt', 'reader-control-random-r1-500224.run.json', 500224)
_cksha['random'] = _h
_mk('random', _st, RandomPLE(_proto['calibrated_head_means'], _proto['calibrated_head_stds'], seed=0))
del _st
_st, _h = _load_state('reader-control-permuted-r1-500224.pt', 'reader-control-permuted-r1-500224.run.json', 500224)
_cksha['permuted'] = _h
_mk('permuted', _st, PermutedPLE(_cperm, seed=777))
del _st
_st, _h = _load_sf('real-500k-r1.reader.safetensors', 'real-500k-r1.run.json', 500224)
_cksha['real'] = _h
_mk('real', _st, _cval)
del _st
_st, _h = _load_sf('real-1m-r1.reader.safetensors', 'real-1m-r1.run.json', 1000448)
_cksha['real-1m'] = _h
_mk('real-1m', _st, _cval)
del _st
_mark('arms: all 5 ready (4x500K controlled + REAL-1M exploratory)')

# Inference primitives: identical tokenization/precision/seeds for all arms. Logprob only.
tokenizer.padding_side = 'left'
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = C.EOS

def _solo(active, mem):
    for _o in _arms:
        _o['inj'].set_memory(mem if _o is active else None)

def _logprob_opts(prompt, conts, arm):
    p = tokenizer(prompt, return_tensors='pt', add_special_tokens=False)['input_ids']
    outs = []
    with torch.inference_mode():
        for c in conts:
            t = tokenizer(c if c.startswith(' ') else ' ' + c, return_tensors='pt', add_special_tokens=False)['input_ids']
            ids = torch.cat([p, t], dim=1)
            _solo(arm, arm['store'].lookup(addresses(ids)).to('cuda') if arm['store'] is not None else None)
            lg = model(input_ids=ids.to('cuda'), use_cache=False).logits.float()
            lp = torch.log_softmax(lg[0, p.shape[1] - 1:-1], dim=-1)
            outs.append(lp.gather(1, t[0].to('cuda').unsqueeze(1)).sum().item())
    return outs

def _lambada_score(context, target, arm):
    p = tokenizer(context, return_tensors='pt', add_special_tokens=False)['input_ids']
    t = tokenizer(target if target.startswith(' ') else ' ' + target, return_tensors='pt', add_special_tokens=False)['input_ids']
    ids = torch.cat([p, t], dim=1)
    with torch.inference_mode():
        _solo(arm, arm['store'].lookup(addresses(ids)).to('cuda') if arm['store'] is not None else None)
        lg = model(input_ids=ids.to('cuda'), use_cache=False).logits.float()
        lp = torch.log_softmax(lg[0, p.shape[1] - 1:-1], dim=-1)
        nll = -lp.gather(1, t[0].to('cuda').unsqueeze(1)).sum().item()
        pred = lp.argmax(-1).cpu()
        acc = int(bool((pred == t[0]).all()))
    return nll, acc, int(t.shape[1])

# Eval determinism: identical logprobs on repeat (disabled arm, 20 prompts).
_det = ['The capital of France is'] * 20
_d1 = _logprob_opts(_det[0], [' Paris', ' London'], _arms[0])
_d2 = _logprob_opts(_det[0], [' Paris', ' London'], _arms[0])
assert _d1 == _d2, 'eval path not deterministic'
print('eval determinism ok', flush=True)
_mark('inference primitives ready')

# Projection: probe per-option rate on 20 examples of the first available MC task + 4 blocks of the first available domain.
_probe_rows = hs_rows or pi_rows or arc_rows or lam_rows
_probe_dom = dom_blocks['general'] if 'general' in dom_blocks else (next(iter(dom_blocks.values())) if dom_blocks else None)
_t = time.perf_counter()
for _r in _probe_rows[:20]:
    _ctx = _r.get('ctx', _r.get('goal', _r.get('question', _r.get('context'))))
    _opts = _r.get('endings', _r.get('sols', _r.get('options', [' ' + _r.get('target', 'x')])))
    _logprob_opts(_ctx, [' ' + o for o in _opts], _arms[0])
_mc_rate = (time.perf_counter() - _t) / max(1, min(20, len(_probe_rows))) if _probe_rows else 0.0
_t = time.perf_counter()
if _probe_dom is not None:
    with torch.inference_mode():
        for _bi in range(min(4, _probe_dom.shape[0])):
            _b = _probe_dom[_bi].unsqueeze(0)
            _solo(_arms[0], None)
            _lg = model(input_ids=_b.to('cuda'), use_cache=False).logits
    _dom_rate = (time.perf_counter() - _t) / 4.0
else:
    _dom_rate = 0.0
_n_mc = len(hs_rows) + len(pi_rows) + len(arc_rows) + len(lam_rows)
_n_dom = sum(dom_blocks[_d].shape[0] for _d in dom_blocks)
proj = (time.perf_counter() - T0) + 5 * (_n_mc * 4 * _mc_rate / 4.0 + _n_dom * _dom_rate)
print('rates: mc %.3fs/4opts dom %.3fs/block | projected total %.0fs (target %d)' % (_mc_rate * 4, _dom_rate, proj, C.EVAL_TARGET_S), flush=True)
MN = C.EVAL_MC_N
while proj > C.EVAL_TARGET_S and MN > C.EVAL_MC_MIN:
    MN //= 2
    _n_mc = 4 * MN
    proj = (time.perf_counter() - T0) + 5 * (_n_mc * _mc_rate + _n_dom * _dom_rate)
print('subsets: MC per-task %d (projected %.0fs)' % (MN, proj), flush=True)
hs_rows, pi_rows, arc_rows, lam_rows = hs_rows[:MN], pi_rows[:MN], arc_rows[:MN], lam_rows[:MN]

results = {'arms': [a['name'] for a in _arms], 'mc': {}, 'domains': {}}
import random as _r
def _boot_diff(vals_a, vals_b, n_boot=10000, seed=1234):
    dd = [a - b for a, b in zip(vals_a, vals_b)]
    rng = _r.Random(seed)
    n = len(dd)
    reps = sorted(sum(dd[rng.randrange(n)] for _ in range(n)) / n for _ in range(n_boot))
    ge = sum(1 for x in reps if x >= 0.0)
    le = sum(1 for x in reps if x <= 0.0)
    return {'mean': sum(dd) / n, 'lo': reps[int(0.025 * n_boot)], 'hi': reps[int(0.975 * n_boot) - 1],
            'p': min(1.0, 2.0 * min(ge, le) / n_boot)}

def _score_mc_task(task, rows, get_prompt, get_opts):
    recs = []
    acc = {a['name']: [] for a in _arms}
    for j, r in enumerate(rows):
        if (time.perf_counter() - T0) > C.EVAL_HARD_S - 1800:
            print('HARD-GUARD: stopping MC early; persisting partial', flush=True)
            break
        prompt = get_prompt(r)
        opts = get_opts(r)
        ns = [tokenizer(o if o.startswith(' ') else ' ' + o, add_special_tokens=False)['input_ids'].__len__() for o in opts]
        rec = {'task': task, 'idx': j, 'label': r['label'], 'n_opts': len(opts), 'opt_lens': ns}
        for a in _arms:
            lps = _logprob_opts(prompt, opts, a)
            pred = max(range(len(opts)), key=lambda i: lps[i])
            pred_n = max(range(len(opts)), key=lambda i: lps[i] / max(1, ns[i]))
            ok = int(pred == r['label'])
            ok_n = int(pred_n == r['label'])
            acc[a['name']].append(ok)
            rec[a['name']] = {'logprobs': lps, 'pred': pred, 'correct': ok, 'pred_norm': pred_n, 'correct_norm': ok_n}
        recs.append(rec)
    (EOUT / f'{task}.jsonl').write_text('\n'.join(json.dumps(x) for x in recs))
    summ = {name: {'acc': sum(v) / max(1, len(v)), 'n': len(v)} for name, v in acc.items()}
    for a in _arms:
        summ[a['name']]['acc_norm'] = sum(x[a['name']]['correct_norm'] for x in recs) / max(1, len(recs))
    results['mc'][task] = {'scores': summ, 'n': len(recs)}
    print('%s: ' % task + ' '.join('%s %.3f' % (k, v['acc']) for k, v in summ.items()), flush=True)
    return recs

_hs = _score_mc_task('hellaswag', hs_rows, lambda r: r['ctx'], lambda r: r['endings'])
_mark('hellaswag done')
_pi = _score_mc_task('piqa', pi_rows, lambda r: r['goal'], lambda r: r['sols'])
_mark('piqa done')
_arc = _score_mc_task('arc-easy', arc_rows, lambda r: r['question'], lambda r: r['options'])
_mark('arc-easy done')

lam_recs = []
lam_acc = {a['name']: [] for a in _arms}
lam_nll = {a['name']: [] for a in _arms}
for j, r in enumerate(lam_rows):
    if (time.perf_counter() - T0) > C.EVAL_HARD_S - 1800:
        print('HARD-GUARD: stopping LAMBADA early; persisting partial', flush=True)
        break
    rec = {'task': 'lambada', 'idx': j, 'target': r['target']}
    for a in _arms:
        nll, acc, nt = _lambada_score(r['context'], r['target'], a)
        lam_acc[a['name']].append(acc)
        lam_nll[a['name']].append(nll / max(1, nt))
        rec[a['name']] = {'nll': nll, 'nll_per_tok': nll / max(1, nt), 'correct': acc, 'ntok': nt}
    lam_recs.append(rec)
(EOUT / 'lambada.jsonl').write_text('\n'.join(json.dumps(x) for x in lam_recs))
results['mc']['lambada'] = {'scores': {name: {'acc': sum(v) / max(1, len(v)), 'nll_per_tok': sum(lam_nll[name]) / max(1, len(lam_nll[name])), 'n': len(v)} for name, v in lam_acc.items()}, 'n': len(lam_recs)}
_mark('lambada done')

for _d in list(dom_blocks):
    if (time.perf_counter() - T0) > C.EVAL_HARD_S - 1800:
        print('HARD-GUARD: stopping before domain %s; persisting partial' % _d, flush=True)
        break
    _blocks = dom_blocks[_d]
    recs = []
    with torch.inference_mode():
        for bi in range(_blocks.shape[0]):
            b = _blocks[bi].unsqueeze(0)
            rec = {'domain': _d, 'block': bi, 'n': int(b[:, 1:].numel())}
            for a in _arms:
                _solo(a, a['store'].lookup(addresses(b.cpu())).to('cuda') if a['store'] is not None else None)
                lg = model(input_ids=b.to('cuda'), use_cache=False).logits
                rec[a['name']] = F.cross_entropy(lg[:, :-1].float().reshape(-1, lg.shape[-1]),
                                                 b.to('cuda')[:, 1:].reshape(-1), reduction='sum').item()
            recs.append(rec)
    assert len(recs) == _blocks.shape[0] and all(set(x) == {'domain', 'block', 'n'} | {a['name'] for a in _arms} for x in recs)
    (EOUT / f"domain-{_d}.jsonl").write_text('\n'.join(json.dumps(x) for x in recs))
    means = {a['name']: sum(x[a['name']] / x['n'] for x in recs) / len(recs) for a in _arms}
    results['domains'][_d] = {'nll': means, 'n': len(recs), 'sha256': dom_metas[_d]['tokens_sha256']}
    print('domain %s: ' % _d + ' '.join('%s %.4f' % (k, v) for k, v in means.items()), flush=True)
    _mark('domain %s done' % _d)

# Paired bootstrap 95% CIs: primary REAL vs 3x500K controls, exploratory REAL-1M vs REAL.
results['mc_errors'] = _mc_err
results['domain_errors'] = dom_err
CONTRASTS = [('real', 'disabled'), ('real', 'random'), ('real', 'permuted'), ('real-1m', 'real')]
boot = {}
for _task, _recs, _key in [('hellaswag', _hs, 'correct'), ('piqa', _pi, 'correct'), ('arc-easy', _arc, 'correct'), ('lambada', lam_recs, 'correct')]:
    boot[_task] = {}
    if not _recs:
        boot[_task]['status'] = 'skipped-empty'
        print('%s: no rows (dataset unresolvable); skipped' % _task, flush=True)
        continue
    for a, b in CONTRASTS:
        va = [x[a][_key] for x in _recs]
        vb = [x[b][_key] for x in _recs]
        m = _boot_diff(va, vb)
        boot[_task]['%s-vs-%s' % (a, b)] = m
        print('%s %s-vs-%s: dacc %+.4f 95%%CI [%+.4f, %+.4f] p=%.4g' % (_task, a.upper(), b.upper(), m['mean'], m['lo'], m['hi'], m['p']), flush=True)
for _d in results['domains']:
    recs = [json.loads(x) for x in (EOUT / f"domain-{_d}.jsonl").read_text().splitlines()]
    boot['domain-' + _d] = {}
    for a, b in CONTRASTS:
        dd = [(x[a] - x[b]) / x['n'] for x in recs]
        rng = _r.Random(1234)
        n = len(dd)
        reps = sorted(sum(dd[rng.randrange(n)] for _ in range(n)) / n for _ in range(10000))
        ge = sum(1 for x in reps if x >= 0.0)
        le = sum(1 for x in reps if x <= 0.0)
        m = {'mean': sum(dd) / n, 'lo': reps[int(0.025 * 10000)], 'hi': reps[int(0.975 * 10000) - 1], 'p': min(1.0, 2.0 * min(ge, le) / 10000)}
        boot['domain-' + _d]['%s-vs-%s' % (a, b)] = m
        print('domain-%s %s-vs-%s: mean %+.5f 95%%CI [%+.5f, %+.5f] p=%.4g %s' % (_d, a.upper(), b.upper(), m['mean'], m['lo'], m['hi'], m['p'], 'ROBUST' if m['hi'] < 0 else 'NOT-ROBUST'), flush=True)
(EOUT / 'bootstrap.json').write_text(json.dumps(boot, indent=2))

(EOUT / 'config.json').write_text(json.dumps({
    'seeds': {'eval': C.EVAL_SEED, 'ple': C.SEED}, 'batch': C.EVAL_BS, 'subsets': {'mc_per_task': MN, 'domain_tokens': C.EVAL_DOMAIN_TOKENS},
    'decoding': 'none; teacher-forced logprob only', 'precision': 'backbone-fp16 reader-FP32 ple-FP32', 'arms': results['arms'],
    'prompts': 'hellaswag ctx/endings; piqa goal/sols; arc question/options; lambada context/target; base model, no chat template',
    'mc_meta': {'hellaswag': hs_meta, 'piqa': pi_meta, 'arc-easy': arc_meta, 'lambada': lam_meta},
    'mc_errors': _mc_err, 'domain_meta': dom_metas, 'domain_errors': dom_err}, indent=2))
(EOUT / 'checkpoint-shas.json').write_text(json.dumps(_cksha, indent=2))
(EOUT / 'timings.json').write_text(json.dumps(timings, indent=2))
(EOUT / 'summary.json').write_text(json.dumps(results, indent=2))
for a in _arms:
    a['inj'].close()
_mark('STAGE A2 COMPLETE: persisted per-example MC, per-block domain NLLs, configs, SHAs, timings, bootstrap')
print('NO training ran in this job (eval-only kernel). Heavy suites deferred per protocol.', flush=True)
